# 02 — Speech-to-Text (ASR) and Transcription Accuracy

**Goal:** turn each call's audio into text with Whisper, then measure how accurate that text is.

**Runs on:** Google Colab with a GPU (Runtime → Change runtime type → T4 GPU). It also runs on a CPU, just much slower.

**Input:** the 25 calls in `results/subset_ids.csv` (two audio files per call: one per speaker).

**Output**
- `results/asr_segments.csv` — every Whisper segment: call, channel, speaker role, start/end time, text, confidence
- `results/asr_words.csv` — every recognized word with its own timestamps (used for speaker turns in notebook 03)
- `results/asr_model_comparison.csv` — the model-size comparison
- `results/asr_vad_experiment.csv` — the quiet-audio experiment
- `results/wer_per_call.csv`, `results/wer_summary.csv` — final accuracy results

**Important:** the human transcripts in the dataset are used **only as the reference** to score Whisper. They are never used as our output.

**Sections**
1. Setup
2. Load the subset and the reference text
3. Text normalization check
4. Baseline: the dataset's own (older) ASR
5. Choose the Whisper model size
6. First run: transcribe all calls with default settings
7. Accuracy of the first run (WER)
8. Error examples
9. Experiment: quiet customer audio
10. Final run and final accuracy
11. Leaked or hallucinated speech check
12. Findings
13. Save and download the results

## 1. Setup

On Colab, this cell clones the repository, installs the ASR libraries and downloads the data. On a laptop it does nothing (run `scripts/download_data.py` yourself).

In [1]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ahmedalmoutiry-wq/call-center-agent-performance-analysis.git"

if IN_COLAB:
    if not os.path.exists("/content/call-center-agent-performance-analysis"):
        !git clone -q {REPO_URL} /content/call-center-agent-performance-analysis
    os.chdir("/content/call-center-agent-performance-analysis")
    # av is pinned: av 19+ removed an argument that faster-whisper 1.2.1 still uses
    !pip install -q faster-whisper==1.2.1 av==18.1.0 jiwer==4.0.0 num2words==0.5.14
    # CUDA 12 libraries for CTranslate2 (newer Colab images ship a newer CUDA version)
    !pip install -q nvidia-cublas-cu12==12.4.5.8 nvidia-cudnn-cu12==9.1.0.70
    !python scripts/download_data.py
    !python scripts/download_data.py --audio results/subset_ids.csv
    !nvidia-smi --query-gpu=name,memory.total --format=csv

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Listing call IDs from GitHub ...
  250/2892 files
  500/2892 files
  750/2892 files
  1000/2892 files
  1250/2892 files
  1500/2892 files
  1750/2892 files
  2000/2892 files
  2250/2892 files
  2500/2892 files
  2750/2892 files
  2892/2892 files

Done. Files are in data/raw/
  50/50 files

Done. Files are in data/raw/
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
sys.path.insert(0, str(PROJECT_ROOT))

from src.asr import ASR_SETTINGS, load_whisper, preload_cuda12_libraries, transcribe_file
from src.data_loading import RESULTS_DIR, audio_path, build_utterances_table
from src.evaluation import overlap_seconds, reference_text, word_error_rate
from src.text_normalization import check_normalization, normalize_text

pd.set_option("display.max_colwidth", 120)
print("Project root:", PROJECT_ROOT)
print("ASR settings:", ASR_SETTINGS)
print("CUDA 12 libraries loaded:", preload_cuda12_libraries())

Project root: /content/call-center-agent-performance-analysis
ASR settings: {'language': 'en', 'beam_size': 5, 'vad_filter': True, 'condition_on_previous_text': False, 'word_timestamps': True}
CUDA 12 libraries loaded: 11


## 2. Load the subset and the reference text

For every call we have **two channels** (two audio files). The speaker role of each channel comes from the transcript, not from the folder name (see notebook 01, section 3.1).

In [3]:
subset = pd.read_csv(RESULTS_DIR / "subset_ids.csv")
utts = build_utterances_table(subset["call_id"].tolist())
speech = utts[~utts["is_empty"] & ~utts["is_noise_only"]]   # reference segments that contain words

# One row per (call, channel): which role is on it, where its audio is, and its reference text
channels = (utts.groupby(["call_id", "channel_index"])["speaker_role"].first()
                .reset_index())
channels["audio_file"] = [audio_path(c, ch) for c, ch in zip(channels["call_id"], channels["channel_index"])]
channels["reference"] = [reference_text(speech[(speech["call_id"] == c) & (speech["channel_index"] == ch)])
                         for c, ch in zip(channels["call_id"], channels["channel_index"])]
channels["audio_exists"] = channels["audio_file"].apply(lambda p: p.exists())

print(f"Calls: {channels['call_id'].nunique()}  |  audio files: {len(channels)}  |  "
      f"missing audio: {(~channels['audio_exists']).sum()}")
channels[["call_id", "channel_index", "speaker_role", "audio_exists", "reference"]].head(4)

Calls: 25  |  audio files: 50  |  missing audio: 0


,call_id,channel_index,speaker_role,audio_exists,reference
0,08f360b3f3b844ef,1,caller,True,hi my name is linda smith i need to check my account balance linda smith all is well i'd like to check my checking a...
1,08f360b3f3b844ef,2,agent,True,hello this is harper valley national bank my name is robert how can i help you today sure i'd be happy to do that fo...
2,09fc75fc02ea4b46,1,caller,True,hi my name is michael jones i need a new checkbook my address is seven three four main street harper valley oregon o...
3,09fc75fc02ea4b46,2,agent,True,hello this is harper valley national bank my name is john how can i help you today what is your address mhm a new ch...


## 3. Text normalization check

WER compares words exactly. Without normalization, `1:30 PM` and `one thirty p m` would be counted as errors although they mean the same. Both reference and Whisper output go through **the same** rules in `src/text_normalization.py`:

- lower case, no punctuation
- numbers, times and amounts written as words (the reference style)
- remove annotation tags (`[noise]`, `<unk>`), word fragments (`wou~`) and hesitation sounds (`uh`, `um`)

The examples below are also tests: the cell fails if any rule breaks.

In [4]:
check_normalization()
from src.text_normalization import NORMALIZATION_EXAMPLES
pd.DataFrame([(raw, normalize_text(raw)) for raw, _ in NORMALIZATION_EXAMPLES],
             columns=["raw text", "normalized"])

,raw text,normalized
0,"Hello, this is Harper Valley National Bank.",hello this is harper valley national bank
1,uh i wou~ i need [noise] <unk> help,i i need help
2,$146 has been transferred.,one hundred and forty six dollars has been transferred
3,Thursday at 1:30 p.m.,thursday at one thirty p m
4,11 AM,eleven a m
5,11am,eleven a m
6,"Okay, all right.",okay alright
7,It's the 3rd.,it's the third
8,My phone number is 0303890619.,my phone number is zero three zero three eight nine zero six one nine
9,It is 303-890-619,it is three zero three eight nine zero six one nine


## 4. Baseline: the dataset's own ASR

The dataset includes an older machine transcript (`provider_asr`). Scoring it on the same calls gives a **reference point** for our Whisper results. It is not our output.

In [5]:
provider = utts.groupby(["call_id", "channel_index"]).apply(
    lambda g: " ".join(g.sort_values("start_s")["provider_asr"]), include_groups=False
).rename("provider_asr").reset_index()
baseline_input = channels.merge(provider, on=["call_id", "channel_index"])

baseline_rows = [{"scope": "baseline (dataset ASR): overall",
                  **word_error_rate(baseline_input["reference"].tolist(), baseline_input["provider_asr"].tolist())}]
for role, group in baseline_input.groupby("speaker_role"):
    baseline_rows.append({"scope": f"baseline (dataset ASR): {role}",
                          **word_error_rate(group["reference"].tolist(), group["provider_asr"].tolist())})
baseline = pd.DataFrame(baseline_rows)
baseline

,scope,wer,ref_words,substitutions,deletions,insertions,n_pairs
0,baseline (dataset ASR): overall,0.0748,2365,104,20,53,50
1,baseline (dataset ASR): agent,0.0688,1467,62,9,30,25
2,baseline (dataset ASR): caller,0.0846,898,42,11,23,25


## 5. Choose the Whisper model size

Bigger Whisper models are usually more accurate but slower. Instead of guessing, we compare three sizes on 3 calls (6 audio files, one call from each of three different task types) and pick the **lowest WER**.

- `small.en`, `medium.en` — English-only models
- `large-v3` — the largest multilingual model

**Real-time factor (RTF)** = processing time ÷ audio duration. RTF 0.1 means one minute of audio takes 6 seconds.

In [6]:
MODELS_TO_COMPARE = ["small.en", "medium.en", "large-v3"]
comparison_calls = subset.drop_duplicates("task_type")["call_id"].head(3).tolist()
comparison_files = channels[channels["call_id"].isin(comparison_calls)]
comparison_audio_s = subset.set_index("call_id").loc[comparison_calls, "duration_s"].sum() * 2

results = []
for model_size in MODELS_TO_COMPARE:
    model, device = load_whisper(model_size)
    hyps, seconds = [], 0.0
    for audio_file in comparison_files["audio_file"]:
        segs, _words, took = transcribe_file(model, audio_file)
        hyps.append(" ".join(s["asr_text"] for s in segs))
        seconds += took
    score = word_error_rate(comparison_files["reference"].tolist(), hyps)
    results.append({"model": model_size, "device": device, "wer": score["wer"],
                    "seconds": round(seconds, 1), "rtf": round(seconds / comparison_audio_s, 3)})
    del model                                   # free GPU memory before loading the next model

model_comparison = pd.DataFrame(results)
model_comparison.to_csv(RESULTS_DIR / "asr_model_comparison.csv", index=False)
print("Calls used:", comparison_calls)
model_comparison

Calls used: ['08f360b3f3b844ef', '37a7a35a1e244820', '09fc75fc02ea4b46']


,model,device,wer,seconds,rtf
0,small.en,cuda/float16,0.0468,4.7,0.012
1,medium.en,cuda/float16,0.0288,7.0,0.018
2,large-v3,cuda/float16,0.0432,10.8,0.028


In [7]:
# Rule: the model with the lowest WER on the comparison calls
CHOSEN_MODEL = model_comparison.sort_values(["wer", "seconds"]).iloc[0]["model"]
print("Chosen model:", CHOSEN_MODEL)

Chosen model: medium.en


## 6. First run: transcribe all calls with default settings

Each audio file is transcribed separately, so every Whisper segment already belongs to one speaker. The role comes from the channel table built in section 2.

Two helper functions are defined here and reused later, so the first run and the final run use exactly the same code:
- `transcribe_channels` — transcribe a set of audio files
- `score` — compute WER overall, by role and per call

In [8]:
model, device = load_whisper(CHOSEN_MODEL)
print(f"Model: {CHOSEN_MODEL} on {device}")

def transcribe_channels(files, **overrides):
    """Transcribe every row of `files`. Return (segments table, words table, seconds taken)."""
    segment_rows, word_rows, seconds = [], [], 0.0
    for ch in files.itertuples():
        segs, words, took = transcribe_file(model, ch.audio_file, **overrides)
        seconds += took
        info = {"call_id": ch.call_id, "channel_index": ch.channel_index, "speaker_role": ch.speaker_role}
        segment_rows += [{**info, **seg} for seg in segs]
        word_rows += [{**info, **w} for w in words]
    segments = pd.DataFrame(segment_rows).sort_values(["call_id", "start_s"]).reset_index(drop=True)
    words = pd.DataFrame(word_rows).sort_values(["call_id", "start_s"]).reset_index(drop=True)
    return segments, words, seconds

def score(files, segments):
    """WER for the given files: (scored table, summary by scope, per-call table)."""
    hyp = (segments.groupby(["call_id", "channel_index"])["asr_text"]
           .apply(" ".join).rename("hypothesis").reset_index())
    scored = files.merge(hyp, on=["call_id", "channel_index"], how="left")
    scored["hypothesis"] = scored["hypothesis"].fillna("")          # a channel with no Whisper output
    rows = [{"scope": "overall", **word_error_rate(scored["reference"].tolist(), scored["hypothesis"].tolist())}]
    for role, g in scored.groupby("speaker_role"):
        rows.append({"scope": f"role: {role}", **word_error_rate(g["reference"].tolist(), g["hypothesis"].tolist())})
    per_call = pd.DataFrame([{"call_id": r.call_id, "speaker_role": r.speaker_role,
                              **word_error_rate([r.reference], [r.hypothesis])} for r in scored.itertuples()])
    per_call = per_call.merge(subset[["call_id", "task_type", "caller_audio_quality"]], on="call_id")
    return scored, pd.DataFrame(rows), per_call

first_segments, first_words, seconds = transcribe_channels(channels)
total_audio_s = subset["duration_s"].sum() * 2
print(f"Segments: {len(first_segments)}  |  words: {len(first_words)}  |  RTF: {seconds / total_audio_s:.3f}")
first_segments.head(8)

Model: medium.en on cuda/float16
Segments: 209  |  words: 2247  |  RTF: 0.020


,call_id,channel_index,speaker_role,segment_id,start_s,end_s,asr_text,avg_logprob,no_speech_prob
0,08f360b3f3b844ef,2,agent,0,1.87,7.27,"Hello, this is Harbor Valley National Bank. My name is Robert. How can I help you today?",-0.177,0.051
1,08f360b3f3b844ef,1,caller,0,10.83,14.61,Hi. My name is Linda Smith. I need to check my account balance.,-0.167,0.037
2,08f360b3f3b844ef,2,agent,1,14.50,19.48,"Sure, I'd be happy to do that for you. And what's your name, please?",-0.177,0.051
3,08f360b3f3b844ef,2,agent,2,20.82,24.02,"Oh, right. It's Linda Smith. Sorry about that. Linda Smith, right.",-0.177,0.051
4,08f360b3f3b844ef,1,caller,1,23.10,24.14,Linda Smith.,-0.167,0.037
5,08f360b3f3b844ef,2,agent,3,25.32,30.02,"Alright, I'm in. And which account would you like to check? Your savings account or your checking account?",-0.177,0.051
6,08f360b3f3b844ef,1,caller,2,25.76,35.58,"All is well. I'd like to check my checking account, please.",-0.167,0.037
7,08f360b3f3b844ef,2,agent,4,42.26,52.14,Your checking account balance is $90 exactly. Great. Is there anything else I could help you with today?,-0.177,0.051


## 7. Accuracy of the first run (WER)

We report WER three ways:
- **Overall** — all calls and both speakers together
- **By speaker role** — the customer side matters most, because summaries and sentiment depend on it
- **Per call** — to find where Whisper struggles

In [9]:
first_scored, first_summary, first_per_call = score(channels, first_segments)
pd.concat([first_summary, baseline], ignore_index=True)

,scope,wer,ref_words,substitutions,deletions,insertions,n_pairs
0,overall,0.0617,2365,49,69,28,50
1,role: agent,0.0320,1467,23,17,7,25
2,role: caller,0.1102,898,26,52,21,25
3,baseline (dataset ASR): overall,0.0748,2365,104,20,53,50
4,baseline (dataset ASR): agent,0.0688,1467,62,9,30,25
5,baseline (dataset ASR): caller,0.0846,898,42,11,23,25


In [10]:
first_per_call.sort_values("wer", ascending=False).head(8)

,call_id,speaker_role,wer,ref_words,substitutions,deletions,insertions,n_pairs,task_type,caller_audio_quality
40,c09408163cb14e39,caller,0.8182,33,0,27,0,1,transfer money,5.0
44,cb7bc09bb8d440dc,caller,0.2692,26,0,1,6,1,schedule appointment,3.0
48,f92c9662d77a4645,caller,0.2500,32,4,0,4,1,reset password,4.0
23,7f3d383f8aff498d,caller,0.2069,29,1,5,0,1,check balance,5.0
30,92f32a1665c04695,caller,0.1795,39,2,4,1,1,replace card,5.0
6,0f87dc392db84250,caller,0.1562,32,3,0,2,1,order checks,4.0
24,8b04254fc6b447b1,caller,0.1224,49,2,1,3,1,transfer money,3.0
15,51cb5e31125c4235,agent,0.1200,50,0,4,2,1,get branch hours,5.0


## 8. Error examples

A WER number alone does not show **what** went wrong. Below: the customer side of the worst and the best call, aligned word by word.

In the alignment, `S` = substitution, `D` = deletion, `I` = insertion, `*` = no word on that side.

In [11]:
import jiwer

def show_alignment(scored, call_id, role="caller"):
    row = scored[(scored["call_id"] == call_id) & (scored["speaker_role"] == role)].iloc[0]
    out = jiwer.process_words(normalize_text(row["reference"]), normalize_text(row["hypothesis"]))
    print(f"===== call {call_id}, {role}  (WER {out.wer:.1%}) =====")
    print(jiwer.visualize_alignment(out, show_measures=False, skip_correct=False))

customer_first = first_per_call[first_per_call["speaker_role"] == "caller"].dropna(subset=["wer"]).sort_values("wer")
WORST_CALL = customer_first.iloc[-1]["call_id"]
show_alignment(first_scored, WORST_CALL)
show_alignment(first_scored, customer_first.iloc[0]["call_id"])

===== call c09408163cb14e39, caller  (WER 81.8%) =====
=== SENTENCE 1 ===

REF: hi my name is jennifer rodriguez i would like to transfer money between my accounts the amount is a hundred and fifty nine dollars from my checking account to my savings account no
HYP: hi my name is jennifer rodriguez * ***** **** ** ******** ***** ******* ** ******** *** ****** ** * ******* *** ***** **** ******* **** ** ******** ******* ** ** ******* ******* **
                                      D     D    D  D        D     D       D  D        D   D      D  D D       D   D     D    D       D    D  D        D       D  D  D       D       D  D

===== call 09fc75fc02ea4b46, caller  (WER 0.0%) =====
=== SENTENCE 1 ===

REF: hi my name is michael jones i need a new checkbook my address is seven three four main street harper valley oregon oh one three two five no that was it you too thanks
HYP: hi my name is michael jones i need a new checkbook my address is seven three four main street harper valley oregon 

## 9. Experiment: quiet customer audio

**What the first run showed:** the worst customer transcript is made almost entirely of **deletions**: Whisper heard the first sentence and then nothing.

**Hypothesis:** this customer's audio is very quiet (notebook 01, section 5.2: its loudness during speech was far below the other calls). The **voice activity detector (VAD)** decides which parts of the audio contain speech before Whisper runs. If the speech is quiet, the VAD may label it as silence and drop it.

**Test 1 — VAD alone (no Whisper):** what share of each customer's reference speech does the VAD keep, at different thresholds? A lower threshold keeps more audio, including more noise.

In [12]:
from faster_whisper.audio import decode_audio
from faster_whisper.vad import VadOptions, get_speech_timestamps

customer_files = channels[channels["speaker_role"] == "caller"]
VAD_THRESHOLDS = [0.5, 0.35, 0.2]          # 0.5 is the faster-whisper default

coverage_rows = []
for ch in customer_files.itertuples():
    audio = decode_audio(str(ch.audio_file))               # 16 kHz float samples
    ref = speech[(speech["call_id"] == ch.call_id) & (speech["channel_index"] == ch.channel_index)]
    ref_seconds = (ref["end_s"] - ref["start_s"]).sum()
    row = {"call_id": ch.call_id, "rms": round(float(np.sqrt(np.mean(audio ** 2))), 4)}
    for th in VAD_THRESHOLDS:
        kept = [(t["start"] / 16000, t["end"] / 16000)
                for t in get_speech_timestamps(audio, VadOptions(threshold=th))]
        covered = sum(overlap_seconds(s, e, kept) for s, e in zip(ref["start_s"], ref["end_s"]))
        row[f"kept_at_{th}"] = round(covered / ref_seconds, 2)
    coverage_rows.append(row)

vad_coverage = pd.DataFrame(coverage_rows).sort_values(f"kept_at_{VAD_THRESHOLDS[0]}")
print("Average share of customer speech kept:")
print(vad_coverage[[f"kept_at_{th}" for th in VAD_THRESHOLDS]].mean().round(3).to_string())
vad_coverage.head(5)

Average share of customer speech kept:
kept_at_0.5     0.952
kept_at_0.35    0.989
kept_at_0.2     0.995


,call_id,rms,kept_at_0.5,kept_at_0.35,kept_at_0.2
20,c09408163cb14e39,0.0067,0.31,0.85,0.95
11,7f3d383f8aff498d,0.0195,0.85,0.97,1.00
22,cb7bc09bb8d440dc,0.0189,0.90,0.92,0.94
8,663201e33f1a4e38,0.0704,0.93,1.00,1.00
16,96c25385230d4172,0.0495,0.94,1.00,1.00


**Test 2 — full transcription of all customer files** with each setting, scored with WER. A lower threshold could recover quiet speech (fewer deletions) but could also let noise through (more insertions), so we measure both.

In [13]:
VARIANTS = {f"vad threshold {th}": {"vad_parameters": {"threshold": th}} for th in VAD_THRESHOLDS}
VARIANTS["vad off"] = {"vad_filter": False}

experiment_rows = []
for name, overrides in VARIANTS.items():
    segs, _words, _sec = transcribe_channels(customer_files, **overrides)
    _scored, summary, per_call = score(customer_files, segs)
    worst = per_call.loc[per_call["call_id"] == WORST_CALL, "wer"].iloc[0]
    experiment_rows.append({"variant": name, **summary.iloc[0].drop("scope").to_dict(),
                            "worst_call_wer": worst})

vad_experiment = pd.DataFrame(experiment_rows)
vad_experiment.to_csv(RESULTS_DIR / "asr_vad_experiment.csv", index=False)
vad_experiment

,variant,wer,ref_words,substitutions,deletions,insertions,n_pairs,worst_call_wer
0,vad threshold 0.5,0.1102,898,26,52,21,25,0.8182
1,vad threshold 0.35,0.0824,898,27,23,24,25,0.1818
2,vad threshold 0.2,0.0813,898,29,18,26,25,0.0909
3,vad off,0.1225,898,36,24,50,25,0.0303


In [14]:
# Rule: the variant with the lowest customer WER (ties -> the earlier, more conservative one)
best = vad_experiment.sort_values("wer", kind="stable").iloc[0]
FINAL_OVERRIDES = VARIANTS[best["variant"]]
print("Chosen setting:", best["variant"], FINAL_OVERRIDES)

Chosen setting: vad threshold 0.2 {'vad_parameters': {'threshold': 0.2}}


## 10. Final run and final accuracy

All 50 files are transcribed again with the chosen setting. These are the results used by the rest of the pipeline.

In [15]:
asr_segments, asr_words, seconds = transcribe_channels(channels, **FINAL_OVERRIDES)
asr_segments.insert(0, "asr_model", CHOSEN_MODEL)
asr_words.insert(0, "asr_model", CHOSEN_MODEL)
asr_segments.to_csv(RESULTS_DIR / "asr_segments.csv", index=False)

scored, final_summary, wer_per_call = score(channels, asr_segments)
wer_per_call.to_csv(RESULTS_DIR / "wer_per_call.csv", index=False)

first_summary = first_summary.assign(scope="first run: " + first_summary["scope"])
final_summary = final_summary.assign(scope="final: " + final_summary["scope"])
wer_summary = pd.concat([final_summary, first_summary, baseline], ignore_index=True)
wer_summary.to_csv(RESULTS_DIR / "wer_summary.csv", index=False)
print(f"RTF: {seconds / total_audio_s:.3f}")
wer_summary

RTF: 0.020


,scope,wer,ref_words,substitutions,deletions,insertions,n_pairs
0,final: overall,0.0490,2365,51,33,32,50
1,final: role: agent,0.0293,1467,22,15,6,25
2,final: role: caller,0.0813,898,29,18,26,25
3,first run: overall,0.0617,2365,49,69,28,50
4,first run: role: agent,0.0320,1467,23,17,7,25
5,first run: role: caller,0.1102,898,26,52,21,25
6,baseline (dataset ASR): overall,0.0748,2365,104,20,53,50
7,baseline (dataset ASR): agent,0.0688,1467,62,9,30,25
8,baseline (dataset ASR): caller,0.0846,898,42,11,23,25


In [16]:
show_alignment(scored, WORST_CALL)
wer_per_call.sort_values("wer", ascending=False).head(8)

===== call c09408163cb14e39, caller  (WER 9.1%) =====
=== SENTENCE 1 ===

REF: hi my name is jennifer rodriguez i would like to transfer money between my accounts the amount is   a hundred and fifty nine dollars from my checking account to my savings account no
HYP: hi my name is jennifer rodriguez i would like to transfer money between my accounts the amount is one hundred and fifty nine dollars from my checking account ** my savings account **
                                                                                                         S                                                          D                     D



,call_id,speaker_role,wer,ref_words,substitutions,deletions,insertions,n_pairs,task_type,caller_audio_quality
6,0f87dc392db84250,caller,0.1875,32,4,0,2,1,order checks,4.0
12,45c7998bd5eb4101,caller,0.1818,33,2,0,4,1,schedule appointment,5.0
30,92f32a1665c04695,caller,0.1795,39,2,4,1,1,replace card,5.0
8,37a7a35a1e244820,caller,0.1379,29,3,1,0,1,get branch hours,5.0
46,d3baa07088cf4c76,caller,0.1236,89,2,6,3,1,pay bill,5.0
24,8b04254fc6b447b1,caller,0.1224,49,2,1,3,1,transfer money,3.0
13,45c7998bd5eb4101,agent,0.1143,70,5,3,0,1,schedule appointment,5.0
16,663201e33f1a4e38,caller,0.1111,45,3,0,2,1,transfer money,5.0


## 11. Leaked or hallucinated speech check

Notebook 01 found that some audio files contain the **other speaker faintly** (echo). Whisper may also invent words in silence (hallucination). Both look the same: a word recognized at a moment when **this speaker was not talking** according to the reference timestamps.

We check every **word** (not segment: one Whisper segment can span several sentences and the pauses between them, so its time range is too coarse).

For each recognized word we ask:
1. Was **this speaker** talking at that moment? (inside one of their reference utterances, ± 0.3 s because reference times are approximate)
2. If not, was the **other speaker** talking? Then the word is probably **echo** from the other side.
3. Neither? Then it is a possible **hallucination**, or speech the reference has no timing for.

In [17]:
PAD_S = 0.3   # tolerance around reference utterances, in seconds

def speech_intervals(call_id, channel_index):
    s = speech[(speech["call_id"] == call_id) & (speech["channel_index"] == channel_index)]
    return list(zip(s["start_s"] - PAD_S, s["end_s"] + PAD_S))

def inside_any(t, intervals):
    return any(start <= t <= end for start, end in intervals)

own_flags, other_flags = {}, {}
for (call_id, channel), words in asr_words.groupby(["call_id", "channel_index"], sort=False):
    own = speech_intervals(call_id, channel)
    other = speech_intervals(call_id, 3 - channel)          # channels are 1 and 2
    mid = (words["start_s"] + words["end_s"]) / 2           # middle of each word
    for i, t in zip(words.index, mid):
        own_flags[i], other_flags[i] = inside_any(t, own), inside_any(t, other)

asr_words["in_own_speech"] = pd.Series(own_flags)
asr_words["in_other_speech"] = pd.Series(other_flags)
asr_words["has_text"] = asr_words["word"].apply(normalize_text) != ""   # ignore "Mm-hmm", punctuation ...

outside = asr_words[asr_words["has_text"] & ~asr_words["in_own_speech"]]
outside = outside.assign(likely_cause=outside["in_other_speech"].map({True: "echo of other speaker",
                                                                      False: "possible hallucination"}))
total = int(asr_words["has_text"].sum())
print(f"Recognized words: {total}  |  outside own speech: {len(outside)} ({len(outside) / total:.1%})")
outside.groupby(["speaker_role", "likely_cause"]).size().rename("words").reset_index()

Recognized words: 2251  |  outside own speech: 13 (0.6%)


,speaker_role,likely_cause,words
0,agent,possible hallucination,1
1,caller,echo of other speaker,4
2,caller,possible hallucination,8


In [18]:
# Group neighbouring flagged words into short phrases, to read them easily
phrases = (outside.groupby(["call_id", "speaker_role", "segment_id", "likely_cause"])
                  .agg(start_s=("start_s", "min"), end_s=("end_s", "max"),
                       text=("word", " ".join), mean_probability=("probability", "mean"))
                  .reset_index().sort_values(["call_id", "start_s"]))
phrases.round(2).head(20)

,call_id,speaker_role,segment_id,likely_cause,start_s,end_s,text,mean_probability
0,09fc75fc02ea4b46,caller,5,possible hallucination,43.02,44.28,Oh,0.24
1,663201e33f1a4e38,caller,2,echo of other speaker,74.58,75.06,I,0.14
2,7f3d383f8aff498d,agent,3,possible hallucination,35.76,36.24,$115.,0.80
3,8b04254fc6b447b1,caller,3,possible hallucination,53.60,54.36,You too. Bye.,0.91
4,a35cc473b91f4de0,caller,0,echo of other speaker,3.41,5.09,"Hi,",0.48
5,a35cc473b91f4de0,caller,0,possible hallucination,25.19,27.83,First,0.51
6,c0c8d261058841ac,caller,0,echo of other speaker,20.18,21.38,12,0.47
7,cb7bc09bb8d440dc,caller,1,possible hallucination,47.43,48.03,Bye -bye.,0.76
8,d3baa07088cf4c76,caller,0,echo of other speaker,21.35,21.35,can,0.47
9,d3baa07088cf4c76,caller,3,possible hallucination,41.54,43.58,to,0.87


In [19]:
# Save the word table with the flags, so notebook 03 can use it
asr_words.to_csv(RESULTS_DIR / "asr_words.csv", index=False)

## 12. Findings

**Model choice**
- `medium.en` had the lowest WER on the 3 comparison calls (2.9%, vs 4.7% for `small.en` and 4.3% for `large-v3`). The comparison used only 6 audio files, so small differences between models are uncertain.
- Speed on a T4 GPU: RTF ≈ 0.02 (one minute of audio in about one second).

**First run (default VAD threshold 0.5)**
- WER: overall 6.2%, agent 3.2%, customer 11.0%.
- The worst customer transcript (`c09408163cb14e39`, WER 81.8%) was almost all deletions: Whisper heard the first sentence, then nothing.

**Root cause: quiet customer audio**
- That customer's audio is very quiet (RMS 0.0067, the lowest in the subset; see also notebook 01).
- The voice activity detector (VAD) at the default threshold kept only 31% of that customer's reference speech, so most of it never reached Whisper.

**Experiment (customer files only)**
- Lowering the VAD threshold recovers quiet speech (fewer deletions) but lets more noise through (more insertions):
  - 0.5: WER 11.0% (52 deletions) | 0.35: 8.2% | 0.2: 8.1% (18 deletions) | VAD off: 12.9% (49 insertions).
- The rule picked 0.2, but 0.35 and 0.2 differ by a single word error: they are effectively tied.

**Final results (VAD threshold 0.2)**
- WER: overall **4.9%**, agent **2.9%**, customer **8.1%**.
- Better than the dataset's own ASR on every scope (overall 7.5%, agent 6.9%, customer 8.5%); the customer-side gain is small.
- The worst call improved from 81.8% to 9.1%.

**Measurement fix found through error analysis**
- Phone numbers were first converted as cardinal numbers ("0303890619" → "three hundred and three million ..."), which counted correct digits as errors. Digit strings that start with 0 or have 5+ digits are now read digit by digit, for both reference and hypothesis.

**Echo and hallucination (section 11)**
- Only a small share of recognized words fall outside the speaker's own reference speech (see the count printed in section 11), so echo from the other channel is not a significant error source. Several flagged words are real closing phrases ("You too. Bye.") that have no timing in the reference.

**Limitations**
- The VAD threshold was chosen on the same 25 calls it is evaluated on, so the final WER may be slightly optimistic. A held-out set of calls would confirm it.
- Some number formats are still ambiguous without context ("734 Main Street" is read as "seven hundred and thirty four", while the reference says "seven three four").
- The reference itself contains small errors (for example "nah" where the speaker likely said "nine").
- The data is scripted role-play with clear speech; real call-center audio would likely give higher WER.

## 13. Save and download the results (Colab only)

Colab files disappear when the session ends. This cell zips the result files so you can download them and copy them into your local project.

Also save the notebook with its outputs: **File → Download → Download .ipynb**.

In [20]:
if IN_COLAB:
    import shutil
    from google.colab import files

    export_dir = Path("/content/asr_results")
    export_dir.mkdir(exist_ok=True)
    for name in ["asr_segments.csv", "asr_words.csv", "asr_model_comparison.csv", "asr_vad_experiment.csv",
                 "wer_summary.csv", "wer_per_call.csv"]:
        shutil.copy(RESULTS_DIR / name, export_dir / name)
    shutil.make_archive("/content/asr_results", "zip", export_dir)
    files.download("/content/asr_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>